# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Drive already mounted at /content/.drive; to attempt to forcibly remount, call drive.mount("/content/.drive", force_remount=True).


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [ ]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle competitions list

ref                                                                           deadline             category         reward  teamCount  userHasEntered  
----------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge   2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                            2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                    2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection          2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False  
https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3                  2026-11-02

In [ ]:
!kaggle competitions submit -c labo-1-virtual-inicial -f "/content/.drive/MyDrive/labo1/exp/exp4024/KA420_32_4032.csv" -m "experimento 4024"

100% 1.73M/1.73M [00:00<00:00, 3.27MB/s]
Successfully submitted to Labo 1 Virtual Inicial

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo1" /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

cp: cannot stat '/content/buckets/b1/kaggle/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory




---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Oct 10 11:31:21 PM 2026"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671290,35.9,1473300,78.7,1473300,78.7
Vcells,1242666,9.5,8388608,64.0,1978689,15.1


In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 600011

# parametros  arbol
# entreno cada arbol con solo 50% de las variables variables
#  por ahora, es fijo
PARAM$feature_fraction <- 0.7

PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 300
PARAM$rpart$minbucket <- 85
PARAM$rpart$maxdepth <- 9

# voy a generar 32 arboles,
#  a mas arboles mas tiempo de proceso y MEJOR MODELO,
#  pero ganancias marginales
PARAM$num_trees_max <- 32

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp4024"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [ ]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

In [ ]:
# que tamanos de ensemble grabo a disco
grabar <- c(32)

In [ ]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
# aqui se va acumulando la probabilidad del ensemble
tb_prediccion[, prob_acumulada := 0]

In [ ]:
set.seed(PARAM$semilla_primigenia) # Establezco la semilla aleatoria

In [ ]:
# contador de combinaciones
iter <- 0

for (vfeature_fraction in c(0.5)) {

  for (vcp in c(-0.5, -0.25)) {

    for (vmaxdepth in c(7, 8)) {

      for (vminsplit in c(175, 225)) {

        for (vminbucket in c(15, 30)) {

# numero de combinación
          iter <- iter + 1
 # RECUPERACION: las iteraciones 1 a 12 ya terminaron
if (iter <= 12) next

          cat(
            "\n------------------------------------",
            "\nIteracion:", iter,
            "\nfeature_fraction:", vfeature_fraction,
            "\ncp:", vcp,
            "\nmaxdepth:", vmaxdepth,
            "\nminsplit:", vminsplit,
            "\nminbucket:", vminbucket,
            "\n------------------------------------\n"
          )

          # asigno los hiperparametros
          PARAM$feature_fraction <- vfeature_fraction

          PARAM$rpart$cp <- vcp
          PARAM$rpart$maxdepth <- vmaxdepth
          PARAM$rpart$minsplit <- vminsplit
          PARAM$rpart$minbucket <- vminbucket

          # reinicio las predicciones para esta combinación
          tb_prediccion <- dfuture[, list(numero_de_cliente)]

          tb_prediccion[, prob_acumulada := 0]

          # misma semilla para cada combinación
          # permite comparar las configuraciones
          set.seed(PARAM$semilla_primigenia)

          # genero los 32 árboles
          for (arbolito in seq(PARAM$num_trees_max)) {

            message(
              "Iteracion ", iter,
              " - Arbol ", arbolito
            )

            # cantidad de variables a utilizar
            qty_campos_a_utilizar <- as.integer(
              length(campos_buenos) *
              PARAM$feature_fraction
            )

            # seleccion aleatoria de variables
            campos_random <- sample(
              campos_buenos,
              qty_campos_a_utilizar
            )

            # genero string para formula
            campos_random <- paste(
              campos_random,
              collapse = " + "
            )

            # formula rpart
            formulita <- paste0(
              "clase_ternaria ~ ",
              campos_random
            )

            # genero árbol
            modelo <- rpart(
              formulita,
              data = dtrain,
              xval = 0,
              control = PARAM$rpart
            )

            # predicción
            prediccion <- predict(
              modelo,
              dfuture,
              type = "prob"
            )

            # acumulo probabilidad BAJA+2
            tb_prediccion[
              ,
              prob_acumulada :=
                prob_acumulada +
                prediccion[, "BAJA+2"]
            ]

            # solamente cuando llego al árbol 32
            if (arbolito %in% grabar) {

              # umbral
              umbral_corte <- (1 / 40) * arbolito

              # predicción final
              tb_prediccion[
                ,
                Predicted :=
                  as.numeric(
                    prob_acumulada > umbral_corte
                  )
              ]

              # nombre único para cada combinación
              archivo_kaggle <- paste0(
                "KA420_032_",
                sprintf("%03d", iter),
                ".csv"
              )

              # guardo archivo
              fwrite(
                tb_prediccion[
                  ,
                  list(numero_de_cliente, Predicted)
                ],
                file = archivo_kaggle,
                sep = ","
              )

              # subida a Kaggle
              comando <- "kaggle competitions submit"

              competencia <- "-c labo-1-virtual-inicial"

              arch <- paste(
                "-f",
                archivo_kaggle
              )

              # descripción de la combinación
              mensaje <- paste0(
                "-m 'iter=", iter,
                " ff=", PARAM$feature_fraction,
                " cp=", PARAM$rpart$cp,
                " minsplit=", PARAM$rpart$minsplit,
                " minbucket=", PARAM$rpart$minbucket,
                " maxdepth=", PARAM$rpart$maxdepth,
                "'"
              )

              linea <- paste(
                comando,
                competencia,
                arch,
                mensaje
              )

              salida <- system(
                linea,
                intern = TRUE
              )

              cat(salida)

            }

          }

        }

      }

    }

  }

}
for (arbolito in seq(PARAM$num_trees_max) ) {
  message( arbolito, " ")
  qty_campos_a_utilizar <- as.integer(length(campos_buenos)
    * PARAM$feature_fraction)

  # elijo los campos al azar
  campos_random <- sample(campos_buenos, qty_campos_a_utilizar)

  # paso de un vector a un string con los elementos
  # separados por un signo de "+"
  # este hace falta para la formula
  campos_random <- paste(campos_random, collapse= " + ")

  # armo la formula para rpart
  formulita <- paste0("clase_ternaria ~ ", campos_random)

  # genero el arbol de decision
  modelo <- rpart(formulita,
    data= dtrain,
    xval= 0,
    control= PARAM$rpart
  )

  # aplico el modelo a los datos que no tienen clase
  prediccion <- predict(modelo, dfuture, type= "prob")

  tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

  if (arbolito %in% grabar) {
    umbral_corte <- (1 / 40) * arbolito
    tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

    archivo_kaggle <- paste0(
        "KA420_32_4",
        sprintf("%.3d", arbolito), # para que tenga ceros adelante
        ".csv"
      )

    # grabo el archivo
    fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
      file= archivo_kaggle,
      sep= ","
    )

    # subida a Kaggle
    comando <- "kaggle competitions submit"
    competencia <- "-c labo-1-virtual-inicial"
    arch <- paste( "-f", archivo_kaggle)

    mensaje <- paste0("-m 'cp=", PARAM$rpart$cp, "  minsplit=", PARAM$rpart$minsplit, "  minbucket=", PARAM$rpart$minbucket, " maxdepth=", PARAM$rpart$maxdepth, "'" )
    linea <- paste( comando, competencia, arch, mensaje)
    salida <- system(linea, intern=TRUE)
    cat(salida)
  }
}



------------------------------------ 
Iteracion: 13 
feature_fraction: 0.5 
cp: -0.25 
maxdepth: 8 
minsplit: 175 
minbucket: 15 
------------------------------------


Iteracion 13 - Arbol 1

Iteracion 13 - Arbol 2

Iteracion 13 - Arbol 3

Iteracion 13 - Arbol 4

Iteracion 13 - Arbol 5

Iteracion 13 - Arbol 6

Iteracion 13 - Arbol 7

Iteracion 13 - Arbol 8

Iteracion 13 - Arbol 9

Iteracion 13 - Arbol 10

Iteracion 13 - Arbol 11

Iteracion 13 - Arbol 12

Iteracion 13 - Arbol 13

Iteracion 13 - Arbol 14

Iteracion 13 - Arbol 15

Iteracion 13 - Arbol 16

Iteracion 13 - Arbol 17

Iteracion 13 - Arbol 18

Iteracion 13 - Arbol 19

Iteracion 13 - Arbol 20

Iteracion 13 - Arbol 21

Iteracion 13 - Arbol 22

Iteracion 13 - Arbol 23

Iteracion 13 - Arbol 24

Iteracion 13 - Arbol 25

Iteracion 13 - Arbol 26

Iteracion 13 - Arbol 27

Iteracion 13 - Arbol 28

Iteracion 13 - Arbol 29

Iteracion 13 - Arbol 30

Iteracion 13 - Arbol 31

Iteracion 13 - Arbol 32

Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit -c labo-1-virtual-inicial -f KA420_032_013.csv -m 'iter=13 ff=0.5 cp=-0.25 minsplit=175 minbucket=15 maxdepth=8'' h

You must authenticate before you can call the Kaggle API. Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
------------------------------------ 
Iteracion: 14 
feature_fraction: 0.5 
cp: -0.25 
maxdepth: 8 
minsplit: 175 
minbucket: 30 
------------------------------------


Iteracion 14 - Arbol 1

Iteracion 14 - Arbol 2

Iteracion 14 - Arbol 3

Iteracion 14 - Arbol 4

Iteracion 14 - Arbol 5

Iteracion 14 - Arbol 6

Iteracion 14 - Arbol 7

Iteracion 14 - Arbol 8

Iteracion 14 - Arbol 9

Iteracion 14 - Arbol 10

Iteracion 14 - Arbol 11

Iteracion 14 - Arbol 12

Iteracion 14 - Arbol 13

Iteracion 14 - Arbol 14

Iteracion 14 - Arbol 15

Iteracion 14 - Arbol 16

Iteracion 14 - Arbol 17

Iteracion 14 - Arbol 18

Iteracion 14 - Arbol 19

Iteracion 14 - Arbol 20

Iteracion 14 - Arbol 21

Iteracion 14 - Arbol 22

Iteracion 14 - Arbol 23

Iteracion 14 - Arbol 24

Iteracion 14 - Arbol 25

Iteracion 14 - Arbol 26

Iteracion 14 - Arbol 27

Iteracion 14 - Arbol 28

Iteracion 14 - Arbol 29

Iteracion 14 - Arbol 30

Iteracion 14 - Arbol 31

Iteracion 14 - Arbol 32

Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit -c labo-1-virtual-inicial -f KA420_032_014.csv -m 'iter=14 ff=0.5 cp=-0.25 minsplit=175 minbucket=30 maxdepth=8'' h

You must authenticate before you can call the Kaggle API. Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
------------------------------------ 
Iteracion: 15 
feature_fraction: 0.5 
cp: -0.25 
maxdepth: 8 
minsplit: 225 
minbucket: 15 
------------------------------------


Iteracion 15 - Arbol 1

Iteracion 15 - Arbol 2

Iteracion 15 - Arbol 3

Iteracion 15 - Arbol 4

Iteracion 15 - Arbol 5

Iteracion 15 - Arbol 6

Iteracion 15 - Arbol 7

Iteracion 15 - Arbol 8

Iteracion 15 - Arbol 9

Iteracion 15 - Arbol 10

Iteracion 15 - Arbol 11

Iteracion 15 - Arbol 12

Iteracion 15 - Arbol 13

Iteracion 15 - Arbol 14

Iteracion 15 - Arbol 15

Iteracion 15 - Arbol 16

Iteracion 15 - Arbol 17

Iteracion 15 - Arbol 18

Iteracion 15 - Arbol 19

Iteracion 15 - Arbol 20

Iteracion 15 - Arbol 21

Iteracion 15 - Arbol 22

Iteracion 15 - Arbol 23

Iteracion 15 - Arbol 24

Iteracion 15 - Arbol 25

Iteracion 15 - Arbol 26

Iteracion 15 - Arbol 27

Iteracion 15 - Arbol 28

Iteracion 15 - Arbol 29

Iteracion 15 - Arbol 30

Iteracion 15 - Arbol 31

Iteracion 15 - Arbol 32

Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit -c labo-1-virtual-inicial -f KA420_032_015.csv -m 'iter=15 ff=0.5 cp=-0.25 minsplit=225 minbucket=15 maxdepth=8'' h

You must authenticate before you can call the Kaggle API. Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
------------------------------------ 
Iteracion: 16 
feature_fraction: 0.5 
cp: -0.25 
maxdepth: 8 
minsplit: 225 
minbucket: 30 
------------------------------------


Iteracion 16 - Arbol 1

Iteracion 16 - Arbol 2

Iteracion 16 - Arbol 3

Iteracion 16 - Arbol 4

Iteracion 16 - Arbol 5

Iteracion 16 - Arbol 6

Iteracion 16 - Arbol 7

Iteracion 16 - Arbol 8

Iteracion 16 - Arbol 9

Iteracion 16 - Arbol 10

Iteracion 16 - Arbol 11

Iteracion 16 - Arbol 12

Iteracion 16 - Arbol 13

Iteracion 16 - Arbol 14

Iteracion 16 - Arbol 15

Iteracion 16 - Arbol 16

Iteracion 16 - Arbol 17

Iteracion 16 - Arbol 18

Iteracion 16 - Arbol 19

Iteracion 16 - Arbol 20

Iteracion 16 - Arbol 21

Iteracion 16 - Arbol 22

Iteracion 16 - Arbol 23

Iteracion 16 - Arbol 24

Iteracion 16 - Arbol 25

Iteracion 16 - Arbol 26

Iteracion 16 - Arbol 27

Iteracion 16 - Arbol 28

Iteracion 16 - Arbol 29

Iteracion 16 - Arbol 30

Iteracion 16 - Arbol 31

Iteracion 16 - Arbol 32

Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit -c labo-1-virtual-inicial -f KA420_032_016.csv -m 'iter=16 ff=0.5 cp=-0.25 minsplit=225 minbucket=30 maxdepth=8'' h

You must authenticate before you can call the Kaggle API. Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication

1 

2 

3 

4 

5 

6 

7 

8 

9 

10 

11 

12 

13 

14 

15 

16 

17 

18 

19 

20 

21 

22 

23 

24 

25 

26 

27 

28 

29 

30 

31 

32 

Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit -c labo-1-virtual-inicial -f KA420_32_4032.csv -m 'cp=-0.25  minsplit=225  minbucket=30 maxdepth=8'' had status 1”


You must authenticate before you can call the Kaggle API. Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Oct 11 02:07:06 AM 2026"



---

